# 3-D Avatar & Persona Agent — Practical Notebook

This notebook supports the browser project. It does not replace the frontend app; it gives the trainer a controlled way to inspect the API contract, persona prompt design, latency budget, and expected responses.

Earlier voice work covered speech-to-text and text-to-speech. This practical connects those pieces to a visible avatar state machine.

## What this build demonstrates

The practical flow is:

```text
Typed or recorded input
→ backend route
→ persona prompt
→ LLM response
→ TTS audio
→ avatar cues
→ frontend audio playback and mouth movement
```

The important idea: the avatar is a synchronized interface layer, not a decorative image.

In [ ]:
from pathlib import Path
import json
import os
import requests
from dotenv import load_dotenv

PROJECT_ROOT = Path.cwd()
load_dotenv(PROJECT_ROOT / ".env")

API_BASE = os.getenv("API_BASE", "http://127.0.0.1:8000")
print(f"Using backend: {API_BASE}")

## Check backend health

Start the backend first:

```bash
uvicorn backend.app:app --reload --port 8000
```

In [ ]:
response = requests.get(f"{API_BASE}/health", timeout=10)
print(response.status_code)
print(response.json())

## Inspect persona cards

A persona is not just a name. It includes role, tone, boundaries, and prompt behavior.

In [ ]:
personas = requests.get(f"{API_BASE}/api/personas", timeout=10).json()
print(json.dumps(personas, indent=2))

## Send a typed message

This call returns the assistant text, generated audio URL, avatar cues, and latency. The frontend uses the same contract.

In [ ]:
payload = {
    "message": "Explain the avatar agent pipeline in 5 short sentences.",
    "persona_id": "bia_ai_mentor",
    "include_audio": True,
}

# COST NOTE: In real mode, this makes one short LLM call and one TTS call.
response = requests.post(f"{API_BASE}/api/chat", json=payload, timeout=60)
print(response.status_code)
data = response.json()
print(json.dumps(data, indent=2))

## What to notice in the response

The response includes:

- `assistant_text`: what the avatar says
- `audio_url`: where the browser retrieves the generated audio
- `avatar_cues`: how the frontend should move through thinking/speaking/idle states
- `latency_ms`: one simple metric for the full request/response turn

This is the backend/frontend contract.

In [ ]:
print("Assistant says:")
print(data.get("assistant_text"))
print("\nAudio URL:", data.get("audio_url"))
print("\nAvatar cues:")
for cue in data.get("avatar_cues", []):
    print(cue)

## Latency budget exercise

Avatar agents feel unnatural when the delay between user input and speech output is too high.

Typical delay sources:

```text
STT → LLM → TTS → browser audio decode → avatar animation
```

The current app reports total latency. A production version would measure each stage separately.

In [ ]:
latency = data.get("latency_ms", 0)
if latency < 1500:
    interpretation = "Very responsive for a request/response classroom demo."
elif latency < 4000:
    interpretation = "Acceptable, but streaming would feel better."
else:
    interpretation = "Noticeable delay. Discuss what can be parallelized or streamed."

print(f"Latency: {latency} ms")
print(interpretation)

## Challenge 1 — Create a new persona

Add a new persona in `backend/persona.py`, restart the backend, and confirm it appears in `/api/personas`.

Hint: copy one existing `Persona(...)` block and modify name, role, tone, boundaries, and system prompt.

## Challenge 2 — Add emotion cues

Modify `backend/avatar_cues.py` so short confident answers return `expression='confident'`, while troubleshooting answers return `expression='focused'`.

Hint: inspect keywords in the assistant text, then set the expression field.

## Challenge 3 — Make latency more transparent

Change the backend so the response includes separate timings for:

- LLM generation
- TTS generation
- total request

Hint: use the existing `timed_ms()` helper around each stage.

## Closing summary

You now have a practical avatar agent pipeline:

- backend controls persona, LLM, STT, and TTS;
- frontend renders a 3D talking head;
- Web Audio maps speech amplitude to mouth movement;
- the avatar visibly reflects agent state.

The production upgrade path is realtime audio, viseme-level lip sync, GLB avatars, memory, guardrails, and observability.